<a href="https://colab.research.google.com/github/00015775/learning-lab/blob/learn%2Fpytorch/pytorch/notebooks/st_gcn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ST-GCN

Sign language is not just motion — it is **structured motion**.

Paper: https://arxiv.org/pdf/1801.07455  
GitHub: https://github.com/fmthoker/st-gcn    
Learn Adjacency Matrix: https://www.geeksforgeeks.org/dsa/adjacency-matrix/

<br>

Learns spatial + temporal together   
Older methods:
  * RNN → good temporal, weak spatial
  * CNN → good spatial, weak temporal

ST-GCN:
  * Both at once.


| Type     | What it learns              |
| -------- | --------------------------- |
| Spatial  | Relationship between joints |
| Temporal | How joints move over time   |


<br>


_`ST-GCN` treats the human body as a moving graph and learns how joints interact over space and time to recognize actions._

| Approach              | Uses graph structure? | Uses edges explicitly? |
| --------------------- | --------------------- | ---------------------- |
| MediaPipe + simple NN |  Usually no          | No                      |
| MediaPipe + LSTM      | No                     | No                      |
| ST-GCN                |  Yes                 | Yes                   |

<br>


Without graph:
* _“Here are 33 numbers per frame, learn something.”_

With ST-GCN:
* _“Here is a body with connected joints moving over time.”_

Input: Joint coordinates over time\
Shape: `(N_nodes, C_features, T_frames)`

```
       ┌─────────────────────┐
       │   Graph Convolution │  ← aggregates info from spatial neighbors (edges)
       └─────────────────────┘
                  │
       ┌──────────────────────┐
       │  Temporal Convolution│  ← aggregates info from same joint across frames
       └──────────────────────┘
                  │
       ┌─────────────────────┐
       │    BatchNorm / ReLU │
       └─────────────────────┘
                  │
                Output
```

<br>

* **Input**: For example, Mediapipe Holistic gives `(x, y, z)` per joint over T frames.

* **Graph Conv Layer**: looks at connected joints (edges) in the same frame.

* **Temporal Conv Layer**: looks at the same joint over time.

* Stack multiple layers → gradually capture higher-level spatial-temporal patterns.

* **Final Layer**: global pooling + fully connected → classification (action/sign).

<br>

_Stacking several ST‑GCN blocks lets the network model local hand/pose features at early layers and whole-body motion patterns at deeper layers._


## Is Mediapipe Holistic + NN the same as ST-GCN?

**Mediapipe Holistic**:  
  * detects landmarks: joints, hands, face, pose.
  * gives coordinates for each joint
  * feed `x,y,z` joint positions over time into NN, effectively treats each joint indepedently or relying on sequential modeling to pick up relationships.
  * There is no explicit graph structure, so the network has to _infer spatial relationships implicitly_.


**ST-GCN**:  
  * Explicitly builds a **graph**: nodes=joints, edges=bones + temporal connections.
  * Uses **graph convolutions** that aggregrate information from connected nodes, thus the network _knows_ how joints are connected and moves naturally over time.
  * This helps to learn hand, pose, face patterns + motion dynamics than a naive NN on a raw coordinates.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GraphConv(nn.Module):
  def __init__(self, in_channels, out_channels, A):
    super().__init__()
    # A: adjacency matrix (edges)
    self.A = torch.tensor(A, dtype=torch.float32)
    self.theta = nn.Parameter(torch.randn(in_channels, out_channels))

  def forward(self, x):
    # x shape: (batch, C_in, N_nodes, T_frames)
    # Graph convolution: aggregates neighbor features
    # Multiply adjancency matrix to propogate node info
    x = torch.einsum('vw, bcwt->bctv', self.A, x) # aggregate neighbors
    x = torch.einsum('bcv, co-> bco', x, self.theta) # apply linear transformation
    return x

class STGCNBlock(nn.Module):
  def __init__(self, in_channels, out_channels, A, kernel_size=9):
    super().__init__()
    self.gcn = GraphConv(in_channels, out_channels, A)
    self.tcn = nn.Conv2d(out_channels,
                         out_channels,
                         (1, kernel_size),
                         padding=(0, kernel_size//2))
    self.bn = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.gcn(x) # spatial graph conv
    x = self.tcn(x) # temporal conv
    x = self.bn(x)
    x = self.relu(x)
    return x

## ST-GCN Pipeline Diagram for Sign Language Recognition

```
Video Frames (RGB)
       │
       ▼
┌───────────────────────────────┐
│  Mediapipe Holistic           │
│  - Pose landmarks (33 joints) │
│  - Hands landmarks (21 each)  │
│  - Face landmarks (468 joints)│
└───────────────────────────────┘
       │
       ▼
┌──────────────────────────────────┐
│ Preprocessing:                   │
│ - Normalize coordinates          │
│ - Stack into shape:              │
│   (batch, channels, nodes, T)    │
│   channels = x, y, z (+optional) │
└──────────────────────────────────┘
       │
       ▼
┌───────────────────────────────┐
│ ST‑GCN Block 1                │
│ - Graph Convolution (spatial) │
│ - Temporal Convolution        │
│ - BatchNorm + ReLU            │
└───────────────────────────────┘
       │
       ▼
┌───────────────────────────────┐
│ ST‑GCN Block 2                │
│ ... (stack multiple layers)   │
└───────────────────────────────┘
       │
       ▼
┌───────────────────────────────┐
│ Global Pooling:               │
│ - Average over nodes & frames │
└───────────────────────────────┘
       │
       ▼
┌──────────────────────────────────┐
│ Fully Connected Layer            │
│ - Output: number of sign classes │
└──────────────────────────────────┘
       │
       ▼
Predicted Sign Label
```

<br>

1. Nodes + edges:

    * Nodes = all joints (pose + hands + optional face)
    * Edges = bones (spatial) + temporal connections  

2. Graph Convolution Layer:

    * Aggregates neighbor information via edges → captures pose structure

3. Temporal Convolution Layer:

    * Aggregates information over time → captures motion patterns

4. Stacked ST‑GCN Blocks:

    * Early layers → local patterns (finger shapes)
    * Deeper layers → global patterns (whole hand-arm-body coordination)

5. Global Pooling + FC Layer:

    * Converts graph features to a fixed-length vector
    * Classifier outputs the predicted sign


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, A):
        super().__init__()
        self.A = torch.tensor(A, dtype=torch.float32)  # adjacency matrix
        self.theta = nn.Parameter(torch.randn(in_channels, out_channels))

    def forward(self, x):
        # x shape: (batch, C_in, N_nodes, T_frames)
        x = torch.einsum('vw,bcvt->bcwt', self.A, x)      # aggregate neighbors
        x = torch.einsum('bcvt,co->bovt', x, self.theta)  # apply linear transform
        return x


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, kernel_size=9):
        super().__init__()
        self.gcn = GraphConv(in_channels, out_channels, A)
        self.tcn = nn.Conv2d(out_channels, out_channels,
                              (1, kernel_size),
                              padding=(0, kernel_size//2))
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.gcn(x)
        x = self.tcn(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

In [17]:
class STGCN(nn.Module):
    def __init__(self, num_class, in_channels, A, num_nodes):
        super().__init__()
        self.block1 = STGCNBlock(in_channels, 64, A)
        self.block2 = STGCNBlock(64, 128, A)
        self.block3 = STGCNBlock(128, 256, A)
        self.pool = nn.AdaptiveAvgPool2d((1,1))  # pool over nodes and time
        self.fc = nn.Linear(256, num_class)

    def forward(self, x):
        # x: (batch, C_in, N_nodes, T_frames)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)           # (batch, channels, 1, 1)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc(x)
        return x

In [20]:
# 33 pose + 21 hand + 21 hand = 75 nodes
num_nodes = 75
num_classes = 50  # 50 signs
in_channels = 3   # x, y, z coordinates
T_frames = 30     # number of frames per sequence (sign sentence)

# iden
A = torch.eye(num_nodes)
for i in range(num_nodes-1):
  A[i, i+1] = 1
  A[i+1, i] = 1

model = STGCN(num_classes, in_channels, A, num_nodes)

# dummy input
x = torch.randn(2, in_channels, num_nodes, T_frames)
print(x.shape) # [BATCH=2, C_IN=3, NODES=75, TEMP=30]
# x.shape = (batch, channels, nodes, frames)

with torch.inference_mode():
  out = model(x)
print(out.shape)


torch.Size([2, 3, 75, 30])
torch.Size([2, 50])


/tmp/ipython-input-887370043.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.A = torch.tensor(A, dtype=torch.float32)  # adjacency matrix


In [21]:
out

tensor([[-0.0692,  0.2641,  0.0438,  0.0584,  0.2307,  0.0507,  0.3001, -0.4898,
          0.1204,  0.2756, -0.1778, -0.0562,  0.2640,  0.1296, -0.0900, -0.2088,
         -0.5253, -0.1548, -0.4217, -0.2640,  0.0455,  0.3184, -0.0887,  0.0577,
          0.2814,  0.2987,  0.4992, -0.0782, -0.0185,  0.1609, -0.0455, -0.0784,
         -0.1183, -0.3649,  0.1838, -0.4857,  0.2699, -0.0143, -0.0165, -0.1431,
         -0.0885, -0.3083, -0.1182,  0.0076, -0.5330,  0.0256,  0.1652, -0.0207,
          0.1275,  0.1036],
        [-0.0814,  0.2608,  0.0575,  0.1010,  0.2454,  0.0412,  0.2820, -0.5039,
          0.1305,  0.3104, -0.1767, -0.0716,  0.2891,  0.2026, -0.1183, -0.2066,
         -0.5839, -0.1483, -0.3927, -0.2109,  0.0080,  0.3497, -0.0939,  0.0692,
          0.3233,  0.3188,  0.4850, -0.0829, -0.0267,  0.1415, -0.0300, -0.0948,
         -0.1208, -0.4014,  0.1685, -0.4770,  0.3380, -0.0056,  0.0107, -0.1493,
         -0.1001, -0.3040, -0.1133, -0.0069, -0.5331, -0.0038,  0.2205,  0.0368,
